# Brecha Salarial por Generación — ENOE Q4 2025
**Fuente:** INEGI — Encuesta Nacional de Ocupación y Empleo (ENOE), Q4 2025  
**Objetivo:** Calcular el ingreso promedio ponderado por grupo de edad y sexo, derivar la brecha de género por generación y visualizarla con una gráfica de barras divergentes.

In [9]:
import pandas as pd
import numpy as np
import re
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

## 1. Carga y limpieza del CSV

El archivo exportado por INEGI tiene un formato especial: cada celda contiene tres valores separados por `|`:
- **Valor 1:** Población ocupada (personas)
- **Valor 2:** Coeficiente de variación (%)
- **Valor 3:** Estimación de la población en miles

Las columnas corresponden a combinaciones de **rango salarial × sexo** (Total / Hombre / Mujer).

In [10]:
# ── Parámetros ────────────────────────────────────────────────────────────────
RUTA_CSV = '../data/INEGI_exporta_23_4_2026_15_39_14.csv'  # ajusta si está en otro directorio

# Salario mínimo mensual aprox. vigente Q4 2025 (SMN diario ~$278.80 × 30.4)
SMN = 8_475  # pesos mensuales

# Punto medio de cada rango salarial (en múltiplos de SMN)
RANGOS = {
    'Hasta_1SM':   SMN * 0.5,
    '1_a_2SM':     SMN * 1.5,
    '2_a_3SM':     SMN * 2.5,
    '3_a_5SM':     SMN * 4.0,
    'Mas_5SM':     SMN * 6.5,   # cota superior abierta → estimación conservadora
    # 'Sin_ingreso' y 'No_especificado' se excluyen del promedio ponderado
}

In [11]:
# ── Lectura del archivo crudo ────────────────────────────────────────────────
with open(RUTA_CSV, encoding='latin-1') as f:
    lineas = f.readlines()

# Filas de datos: líneas pares con contenido real (índices 13,15,17,19,21,23,25,27)
# Saltamos 'Total' y 'No especificado'
GRUPOS_EDAD_RAW = [
    (15, ' 15 a 19'),
    (17, ' 20 a 29'),
    (19, ' 30 a 39'),
    (21, ' 40 a 49'),
    (23, ' 50 a 59'),
    (25, ' 60'),
]

def extraer_poblacion(celda: str) -> float:
    """Extrae el primer valor (población) de una celda con formato 'valor|cv|est|'."""
    celda = celda.strip()
    if not celda or celda == '':
        return np.nan
    partes = celda.split('|')
    try:
        return float(partes[0].replace(',', ''))
    except (ValueError, IndexError):
        return np.nan

# Nombre de las 24 columnas (rango × sexo)
COLUMNAS = [
    'Total_Total',    'Total_Hombre',    'Total_Mujer',
    'Hasta1_Total',   'Hasta1_Hombre',   'Hasta1_Mujer',
    '1a2_Total',      '1a2_Hombre',      '1a2_Mujer',
    '2a3_Total',      '2a3_Hombre',      '2a3_Mujer',
    '3a5_Total',      '3a5_Hombre',      '3a5_Mujer',
    'Mas5_Total',     'Mas5_Hombre',     'Mas5_Mujer',
    'SinIng_Total',   'SinIng_Hombre',   'SinIng_Mujer',
    'NoEsp_Total',    'NoEsp_Hombre',    'NoEsp_Mujer',
]

registros = []
for idx_linea, prefijo in GRUPOS_EDAD_RAW:
    linea = lineas[idx_linea]
    partes = linea.split(',')
    nombre_edad = partes[0].strip()
    celdas = partes[1:]  # 24 celdas de datos

    fila = {'Edad': nombre_edad}
    for i, col in enumerate(COLUMNAS):
        fila[col] = extraer_poblacion(celdas[i]) if i < len(celdas) else np.nan
    registros.append(fila)

df_raw = pd.DataFrame(registros)
df_raw.head()

,Edad,Total_Total,Total_Hombre,Total_Mujer,Hasta1_Total,Hasta1_Hombre,Hasta1_Mujer,1a2_Total,1a2_Hombre,1a2_Mujer,...,3a5_Mujer,Mas5_Total,Mas5_Hombre,Mas5_Mujer,SinIng_Total,SinIng_Hombre,SinIng_Mujer,NoEsp_Total,NoEsp_Hombre,NoEsp_Mujer
0,15 a 19 años,2969753.0,2010270.0,959483.0,1725900.0,1113443.0,612457.0,584691.0,461809.0,122882.0,...,4651.0,2171.0,1039.0,1132.0,431054.0,280234.0,150820.0,189493.0,128099.0,61394.0
1,20 a 29 años,12308362.0,7236614.0,5071748.0,4840393.0,2521717.0,2318676.0,4447855.0,2877295.0,1570560.0,...,58528.0,44242.0,29737.0,14505.0,580673.0,313246.0,267427.0,1521968.0,896257.0,625711.0
2,30 a 39 años,14044114.0,7896614.0,6147500.0,4664681.0,2139829.0,2524852.0,5289255.0,3269228.0,2020027.0,...,152373.0,145655.0,105857.0,39798.0,383263.0,183367.0,199896.0,1858397.0,1076961.0,781436.0
3,40 a 49 años,12995804.0,7398022.0,5597782.0,4494653.0,2052806.0,2441847.0,4650964.0,2957024.0,1693940.0,...,121906.0,154170.0,122267.0,31903.0,405020.0,165020.0,240000.0,1797995.0,1102105.0,695890.0
4,50 a 59 años,10791558.0,6421034.0,4370524.0,4185953.0,2080999.0,2104954.0,3624064.0,2409771.0,1214293.0,...,91383.0,107398.0,77088.0,30310.0,396102.0,181042.0,215060.0,1390295.0,900735.0,489560.0


## 2. Cálculo del ingreso promedio ponderado por sexo

Usamos la distribución de población en cada rango salarial para estimar el ingreso promedio:

$$\bar{w} = \frac{\sum_{r} n_{r} \cdot m_{r}}{\sum_{r} n_{r}}$$

donde $n_r$ es la población en el rango $r$ y $m_r$ es el punto medio del rango.

In [12]:
# Mapeo: columna de población → punto medio salarial
# Solo incluimos rangos con ingreso definido (excluimos SinIng y NoEsp)
RANGOS_COLS = {
    'Hasta1': RANGOS['Hasta_1SM'],
    '1a2':    RANGOS['1_a_2SM'],
    '2a3':    RANGOS['2_a_3SM'],
    '3a5':    RANGOS['3_a_5SM'],
    'Mas5':   RANGOS['Mas_5SM'],
}

def ingreso_ponderado(df: pd.DataFrame, sexo: str) -> pd.Series:
    """Calcula el ingreso promedio ponderado para el sexo dado ('Hombre' o 'Mujer')."""
    numerador   = sum(df[f'{rango}_{sexo}'].fillna(0) * mid for rango, mid in RANGOS_COLS.items())
    denominador = sum(df[f'{rango}_{sexo}'].fillna(0) for rango in RANGOS_COLS)
    return numerador / denominador

df_raw['ing_hombres'] = ingreso_ponderado(df_raw, 'Hombre')
df_raw['ing_mujeres'] = ingreso_ponderado(df_raw, 'Mujer')

# Brecha relativa: (H - M) / H  → positivo = hombres ganan más
df_raw['brecha_pct'] = (df_raw['ing_hombres'] - df_raw['ing_mujeres']) / df_raw['ing_hombres'] * 100

# Brecha absoluta en pesos
df_raw['brecha_abs'] = df_raw['ing_hombres'] - df_raw['ing_mujeres']

# Población total por grupo (para escalar burbujas)
df_raw['pob_total'] = df_raw['Total_Total']

df_raw[['Edad','ing_hombres','ing_mujeres','brecha_abs','brecha_pct','pob_total']]

,Edad,ing_hombres,ing_mujeres,brecha_abs,brecha_pct,pob_total
0,15 a 19 años,7017.023118,6032.220626,984.802492,14.034477,2969753.0
1,20 a 29 años,10499.734002,8892.431185,1607.302817,15.308034,12308362.0
2,30 a 39 años,12683.617152,10225.840166,2457.776987,19.377572,14044114.0
3,40 a 49 años,12690.466842,9794.227392,2896.239450,22.822166,12995804.0
4,50 a 59 años,11765.613940,9244.773707,2520.840233,21.425488,10791558.0
5,60 años y más,9868.071688,7465.943946,2402.127742,24.342423,6641460.0


## 3. Asignación de generaciones

Mapeo estándar (usando el año de referencia 2025):

| Grupo de edad | Año de nacimiento aprox. | Generación |
|---|---|---|
| 15–19 | 2006–2010 | Generación Z (jóvenes) |
| 20–29 | 1996–2005 | Generación Z / Millennials |
| 30–39 | 1986–1995 | Millennials |
| 40–49 | 1976–1985 | Generación X |
| 50–59 | 1966–1975 | Baby Boomers tardíos / Gen X |
| 60+ | antes de 1965 | Baby Boomers / Silenciosa |

In [13]:
GENERACIONES = {
    '15 a 19 años':   'Gen Z (15–19)',
    '20 a 29 años':   'Gen Z / Millennials (20–29)',
    '30 a 39 años':   'Millennials (30–39)',
    '40 a 49 años':   'Generación X (40–49)',
    '50 a 59 años':   'Baby Boomers (50–59)',
    '60 años y más':  'Silenciosa / BB senior (60+)',
}

# Colores por generación (alineados con paleta del dashboard)
COLORES_GEN = {
    'Gen Z (15–19)':              '#7c3aed',
    'Gen Z / Millennials (20–29)':'#3f5cda',
    'Millennials (30–39)':        '#2B4C6F',
    'Generación X (40–49)':       '#d97706',
    'Baby Boomers (50–59)':       '#e8517a',
    'Silenciosa / BB senior (60+)':'#6b7280',
}

df_raw['Generacion'] = df_raw['Edad'].map(GENERACIONES)

# DataFrame limpio final
df = df_raw[[
    'Edad', 'Generacion',
    'ing_hombres', 'ing_mujeres',
    'brecha_abs', 'brecha_pct',
    'pob_total'
]].copy()

# Ordenar de joven a mayor
orden = list(GENERACIONES.values())
df['_orden'] = df['Generacion'].map({v: i for i, v in enumerate(orden)})
df = df.sort_values('_orden').drop(columns='_orden').reset_index(drop=True)

print("Dataset limpio:")
df

Dataset limpio:


,Edad,Generacion,ing_hombres,ing_mujeres,brecha_abs,brecha_pct,pob_total
0,15 a 19 años,Gen Z (15–19),7017.023118,6032.220626,984.802492,14.034477,2969753.0
1,20 a 29 años,Gen Z / Millennials (20–29),10499.734002,8892.431185,1607.302817,15.308034,12308362.0
2,30 a 39 años,Millennials (30–39),12683.617152,10225.840166,2457.776987,19.377572,14044114.0
3,40 a 49 años,Generación X (40–49),12690.466842,9794.227392,2896.239450,22.822166,12995804.0
4,50 a 59 años,Baby Boomers (50–59),11765.613940,9244.773707,2520.840233,21.425488,10791558.0
5,60 años y más,Silenciosa / BB senior (60+),9868.071688,7465.943946,2402.127742,24.342423,6641460.0


## 4. Exportar dataset limpio (opcional)

In [14]:
df.to_csv('../data/clean_data/brecha_generacional_clean.csv', index=False, encoding='utf-8-sig')
print("Guardado: brecha_generacional_clean.csv")

Guardado: brecha_generacional_clean.csv


## 5. Gráfica de barras divergentes — Brecha por Generación

La gráfica muestra el ingreso de hombres y mujeres como barras que "divergen" desde el centro del eje Y, haciendo visible la asimetría. El ancho de la barra encarna la magnitud del ingreso; el espacio entre ellas, la brecha.

In [15]:
# ── Estilo del dashboard ──────────────────────────────────────────────────────
COLOR_MASCULINO = '#3f5cda'
COLOR_FEMENINO  = '#e8517a'
COLOR_BRECHA    = '#d97706'
FONT            = dict(family='DM Sans, Arial, sans-serif', size=12)

gens   = df['Generacion'].tolist()
ing_h  = df['ing_hombres'].tolist()
ing_m  = df['ing_mujeres'].tolist()
brecha = df['brecha_pct'].tolist()
brecha_abs = df['brecha_abs'].tolist()

fig = go.Figure()

# ── Barras de mujeres (izquierda, valores negativos) ─────────────────────────
fig.add_trace(go.Bar(
    name='Mujeres',
    y=gens,
    x=[-v for v in ing_m],          # negativo → barra hacia la izquierda
    orientation='h',
    marker=dict(
        color=COLOR_FEMENINO,
        opacity=0.88,
        line=dict(color='white', width=1.2)
    ),
    text=[f'${v:,.0f}' for v in ing_m],
    textposition='inside',
    textfont=dict(color='white', size=10, family='DM Sans, Arial, sans-serif'),
    hovertemplate=(
        '<b>%{y}</b><br>'
        'Ingreso mujeres: $%{customdata[0]:,.0f}<br>'
        'Brecha relativa: %{customdata[1]:.1f}%<extra></extra>'
    ),
    customdata=list(zip(ing_m, brecha)),
))

# ── Barras de hombres (derecha, valores positivos) ────────────────────────────
fig.add_trace(go.Bar(
    name='Hombres',
    y=gens,
    x=ing_h,
    orientation='h',
    marker=dict(
        color=COLOR_MASCULINO,
        opacity=0.88,
        line=dict(color='white', width=1.2)
    ),
    text=[f'${v:,.0f}' for v in ing_h],
    textposition='inside',
    textfont=dict(color='white', size=10, family='DM Sans, Arial, sans-serif'),
    hovertemplate=(
        '<b>%{y}</b><br>'
        'Ingreso hombres: $%{x:,.0f}<br>'
        'Brecha relativa: %{customdata:.1f}%<extra></extra>'
    ),
    customdata=brecha,
))

# ── Anotaciones de brecha sobre cada par de barras ────────────────────────────
for i, (gen, b_pct, b_abs) in enumerate(zip(gens, brecha, brecha_abs)):
    fig.add_annotation(
        x=0, y=gen,
        text=f'  <b>{b_pct:.1f}%</b><br><span style="font-size:9px">−${b_abs:,.0f}</span>',
        showarrow=False,
        font=dict(size=9.5, color=COLOR_BRECHA, family='DM Sans, Arial, sans-serif'),
        xanchor='center',
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor=COLOR_BRECHA,
        borderwidth=1,
        borderpad=3,
    )

# ── Layout ────────────────────────────────────────────────────────────────────
max_val = max(max(ing_h), max(ing_m)) * 1.18

fig.update_layout(
    title=dict(
        text='<b>BRECHA SALARIAL DE GÉNERO POR GENERACIÓN</b><br>'
             '<span style="font-size:11px;color:#666">Ingreso promedio ponderado por rango salarial — ENOE Q4 2025 · $MXN mensuales</span>',
        font=dict(size=14, family='DM Sans, Arial, sans-serif', color='#1a1a2e'),
        x=0.5, xanchor='center',
        pad=dict(b=10)
    ),
    barmode='overlay',
    height=440,
    margin=dict(l=220, r=60, t=90, b=60),
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=FONT,
    legend=dict(
        orientation='h', x=0.5, xanchor='center', y=-0.15,
        font=dict(size=11)
    ),
    xaxis=dict(
        title='Ingreso promedio mensual ponderado ($MXN)',
        tickformat='$,.0f',
        range=[-max_val, max_val],
        tickvals=[-10000, -8000, -6000, -4000, -2000, 0, 2000, 4000, 6000, 8000, 10000],
        ticktext=['$10k', '$8k', '$6k', '$4k', '$2k', '$0', '$2k', '$4k', '$6k', '$8k', '$10k'],
        gridcolor='rgba(0,0,0,0.06)',
        zeroline=True,
        zerolinecolor='rgba(0,0,0,0.3)',
        zerolinewidth=2,
    ),
    yaxis=dict(
        autorange='reversed',
        gridcolor='rgba(0,0,0,0)',
        tickfont=dict(size=10.5)
    ),
    shapes=[
        # Línea central destacada
        dict(
            type='line', x0=0, x1=0, y0=-0.5, y1=len(gens)-0.5,
            line=dict(color='rgba(0,0,0,0.25)', width=1.5, dash='dot'),
            layer='above'
        )
    ]
)

fig.show(config={'displayModeBar': True, 'toImageButtonOptions': {'filename': 'brecha_generacional', 'format': 'png'}})

## 6. Interpretación

El porcentaje encima de cada par de barras es la **brecha relativa** `(H−M)/H`.  
Un valor alto significa que las mujeres perciben significativamente menos que los hombres en ese grupo de edad, independientemente del nivel salarial absoluto.

**¿Está cerrando la brecha la Generación Z?**  
Compare las barras de *Gen Z (15–19)* y *Gen Z / Millennials (20–29)* contra las generaciones mayores. Si la brecha porcentual es menor en los grupos jóvenes, hay evidencia de convergencia. Si es igual o mayor, el sistema está replicando la desigualdad desde el inicio de la trayectoria laboral.